# Explainable AI MRI Analysis and Medical Report Generation System

**Research/educational output — not a medical diagnosis. Results must be reviewed and validated by a qualified radiologist/physician.**

This notebook builds and demonstrates a complete, modular pipeline:

```
MRI (DICOM/NIfTI/PNG/JPG) -> Preprocessing -> Vision Encoder -> Finding Detection
    -> Segmentation -> XAI (Grad-CAM / Integrated Gradients / Attention / Occlusion)
    -> Structured Evidence -> LLM report generation -> Gradio UI
```

Run cells top to bottom on a **GPU Colab runtime** (`Runtime > Change runtime type > GPU`).
Every stage degrades gracefully (with a clear message) if a model download or GPU is unavailable,
so the notebook remains demoable even offline/CPU-only.


## 01 — Environment Setup


In [ ]:
import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())

def is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IN_COLAB = is_colab()
print("Running in Colab:", IN_COLAB)


## 02 — GPU Check


In [ ]:
import subprocess
try:
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("nvidia-smi not found -- no NVIDIA GPU detected in this environment.")


In [ ]:
import torch
CUDA_AVAILABLE = torch.cuda.is_available()
print("CUDA available:", CUDA_AVAILABLE)
if CUDA_AVAILABLE:
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("No GPU detected -- the pipeline will run on CPU (slower; heavy models may be skipped).")
DEVICE = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
print("Using device:", DEVICE)


## 03 — Install Dependencies

Skip this cell if you already `pip install -r requirements.txt` locally.


In [ ]:
%%capture
import sys, subprocess

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "torch", "torchvision", "monai", "nibabel", "pydicom", "opencv-python-headless",
        "transformers", "datasets", "accelerate", "peft", "bitsandbytes", "sentencepiece",
        "captum", "gradio", "pydantic", "pyyaml", "rouge-score", "sacrebleu", "bert-score",
        "nltk", "tensorboard"])
print("Dependency installation complete (or skipped outside Colab).")


In [ ]:
import importlib
required = ["torch","torchvision","monai","nibabel","pydicom","cv2","transformers",
            "peft","captum","gradio","pydantic","yaml","sklearn","scipy","numpy","pandas","matplotlib"]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)
print("All core dependencies importable." if not missing else f"Missing (install manually if needed): {missing}")


## 04 — Project Directory Structure & Configuration

Clones/creates the `mri_xai_llm/` package. In Colab this notebook and the `mri_xai_llm/` folder
should be uploaded together (or cloned from your repo) into `/content/`.


In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/ankitupadhyay662/XAI-MRI-LLM-Model.git"

if "google.colab" in sys.modules:
    PROJECT_ROOT = Path("/content")
else:
    PROJECT_ROOT = Path.cwd()

PKG_DIR = PROJECT_ROOT / "mri_xai_llm"
CLONE_DIR = PROJECT_ROOT / "_mri_xai_llm_repo"

# Always sync PKG_DIR to the latest commit on GitHub -- NOT just "clone if
# missing". `Runtime > Restart session` resets the Python kernel but does
# NOT delete files under /content/, so a stale PKG_DIR from an earlier run
# would otherwise silently shadow every code fix pushed after that point.
if "google.colab" in sys.modules:
    import subprocess, shutil

    if CLONE_DIR.exists():
        result = subprocess.run(
            ["git", "-C", str(CLONE_DIR), "pull", "--ff-only"],
            capture_output=True, text=True,
        )
    else:
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)],
            capture_output=True, text=True,
        )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(
            f"git clone/pull failed (exit {result.returncode}). If the repo is private, "
            "clone manually with a token first: "
            f"!git clone https://<token>@github.com/ankitupadhyay662/XAI-MRI-LLM-Model.git {CLONE_DIR}"
        )

    if PKG_DIR.exists():
        shutil.rmtree(PKG_DIR)
    shutil.copytree(CLONE_DIR / "mri_xai_llm", PKG_DIR)
    head = subprocess.run(
        ["git", "-C", str(CLONE_DIR), "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True,
    ).stdout.strip()
    print(f"Synced {PKG_DIR} to commit {head}")

assert PKG_DIR.exists(), (
    f"Expected the mri_xai_llm/ package at {PKG_DIR}. "
    "Outside Colab: place the mri_xai_llm/ folder next to this notebook manually."
)

for p in (str(PROJECT_ROOT), str(PKG_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

for sub in ["data/raw", "data/processed", "data/sample", "experiments", "checkpoints"]:
    (PKG_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Package dir:", PKG_DIR)

In [ ]:
from config import config as cfg
cfg.set_seed(cfg.SEED)

try:
    cfg.load_yaml_overrides(PKG_DIR / "config.yaml")
    print("Loaded config.yaml overrides.")
except Exception as exc:
    print("config.yaml overrides not applied:", exc)

print("Seed:", cfg.SEED)
print("Model config:", cfg.MODEL_CONFIG)
print("Min confidence threshold:", cfg.MIN_CONFIDENCE)
print("Disclaimer:", cfg.DISCLAIMER)


## 05 — Dataset Loading

Public research datasets are **not** auto-downloaded here (license/access requirements vary).
This cell documents adapters and expects a JSONL manifest if you want to run training/evaluation.

| Dataset | Contents | Reports | Diagnostic labels | Research use |
|---|---|---|---|---|
| BraTS | Brain MRI (T1/T1CE/T2/FLAIR) + tumor segmentation masks | No | Segmentation labels only | Yes, per BraTS DUA |
| IXI | ~600 healthy-subject brain MRIs (T1/T2/PD/MRA/DWI) | No | No | Yes, freely available |
| fastMRI | Knee/brain k-space + reconstructions | No | No | Yes, registration required |
| OpenNeuro | Many open MRI/fMRI datasets | Varies | Varies | Per-dataset license (often CC0) |


In [ ]:
import json

def load_jsonl_manifest(path):
    """Loads a dataset manifest of {image, patient_id, study_id, modality, body_part,
    sequence, report, finding, diagnosis} records. Real patient identifiers must NOT be present."""
    records = []
    p = Path(path)
    if not p.exists():
        print(f"No manifest found at {p} -- skipping dataset loading (expected for a fresh clone).")
        return records
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    print(f"Loaded {len(records)} records from {p}")
    return records

sample_manifest_path = PKG_DIR / "data" / "sample" / "manifest.jsonl"
dataset_records = load_jsonl_manifest(sample_manifest_path)


## 06 — DICOM Loader


In [ ]:
from preprocessing import dicom as dicom_mod

print(dicom_mod.__doc__ or "DICOM loader module ready.")
print("Public functions:", [n for n in dir(dicom_mod) if not n.startswith("_")])


## 07 — NIfTI Loader


In [ ]:
from preprocessing import nifti as nifti_mod

print(nifti_mod.__doc__ or "NIfTI loader module ready.")
print("Public functions:", [n for n in dir(nifti_mod) if not n.startswith("_")])


## 08 — MRI Preprocessing (orientation, normalization, quality, slice selection)


In [ ]:
from preprocessing import transforms as tfm
import numpy as np

# Synthetic demo volume (stand-in until a real MRI is uploaded in the Gradio UI below).
rng = np.random.default_rng(cfg.SEED)
demo_volume = rng.normal(loc=500, scale=80, size=(64, 128, 128)).astype(np.float32)
demo_volume[20:44, 40:90, 40:90] += 150  # synthetic "lesion-like" bright region

norm = tfm.normalize_intensity(demo_volume, method="percentile")
print("Normalized intensity range:", float(norm.min()), float(norm.max()))

quality = tfm.assess_image_quality(demo_volume[32])
print("Image quality assessment (model-derived, not clinically validated):", quality)

slice_idx = tfm.select_representative_slices(demo_volume, num_slices=5, axis=0)
print("Selected representative slice indices:", slice_idx)


## 09 — Visualization


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(slice_idx), figsize=(15, 3))
for ax, idx in zip(axes, slice_idx):
    ax.imshow(demo_volume[idx], cmap="gray")
    ax.set_title(f"slice {idx}")
    ax.axis("off")
plt.suptitle("Selected representative slices (demo synthetic volume)")
plt.tight_layout()
plt.show()


## 10 — Model Loading

Uses `ModelManager` for lazy, swappable, memory-conscious model loading.
Downloads may fail without internet access -- the manager logs a warning and falls back automatically.


In [ ]:
from models import ModelManager

model_manager = ModelManager()
vision_encoder = model_manager.load_vision_model()
print("Vision encoder loaded:", type(vision_encoder).__name__)


## 11 — Feature Extraction


In [ ]:
from models.vision_encoder import preprocess_for_encoder
import torch

demo_slice = demo_volume[32]
input_tensor = preprocess_for_encoder(demo_slice, image_size=cfg.MODEL_CONFIG["image_size"]).to(model_manager.device)

with torch.no_grad():
    features = vision_encoder(input_tensor)

print("Pooled embedding shape:", tuple(features["pooled"].shape))
print("Spatial feature map shape:", tuple(features["spatial"].shape))


## 12 — Finding Detection

**Note:** without a fine-tuned checkpoint, `FindingClassifierHead` is randomly initialized.
Its scores demonstrate the architecture's data flow only -- they are NOT validated predictions.

In [ ]:
from models.multimodal_model import FindingClassifierHead

finding_head = FindingClassifierHead(vision_encoder.hidden_dim).to(model_manager.device)
finding_head.eval()

with torch.no_grad():
    head_out = finding_head(features["pooled"])
    probs = torch.sigmoid(head_out["finding_logits"])[0].cpu().numpy()

taxonomy = finding_head.taxonomy
ranked = sorted(zip(taxonomy, probs), key=lambda x: -x[1])[:5]
for name, score in ranked:
    flag = "" if score >= cfg.MIN_CONFIDENCE else "  <- below MIN_CONFIDENCE, would be flagged uncertain"
    print(f"{name:30s} score={score:.3f}{flag}")


## 13 — Segmentation


In [ ]:
from models.segmentation import LesionSegmentationModel, segment, compute_lesion_metrics

seg_model = LesionSegmentationModel(in_channels=1, out_channels=1).to(model_manager.device)
seg_model.eval()

mask = segment(seg_model, demo_slice)
metrics = compute_lesion_metrics(mask, voxel_spacing=None)
print("Approximate lesion metrics (segmentation model is untrained/demo unless a checkpoint is loaded):")
print(metrics)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(demo_slice, cmap="gray"); axes[0].set_title("Original MRI"); axes[0].axis("off")
axes[1].imshow(mask, cmap="magma"); axes[1].set_title("Segmentation Mask"); axes[1].axis("off")
axes[2].imshow(demo_slice, cmap="gray"); axes[2].imshow(mask, cmap="magma", alpha=0.4); axes[2].set_title("Overlay"); axes[2].axis("off")
plt.tight_layout(); plt.show()


## 14 — XAI: Grad-CAM


In [ ]:
from xai import gradcam

class _Combo(torch.nn.Module):
    def __init__(self, enc, head):
        super().__init__()
        self.enc, self.head = enc, head
    def forward(self, x):
        v = self.enc(x)
        h = self.head(v["pooled"])
        return {"finding_logits": h["finding_logits"], "spatial": v["spatial"]}

combo = _Combo(vision_encoder, finding_head).to(model_manager.device).eval()
top_idx = int(np.argmax(probs))

with gradcam.GradCAM(vision_encoder.get_target_layer()) as cam:
    heatmap = cam.generate(combo, input_tensor, target_class_idx=top_idx)

overlay = gradcam.overlay_heatmap(demo_slice.astype(np.float32), heatmap, alpha=0.4)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(demo_slice, cmap="gray"); axes[0].set_title("Original MRI"); axes[0].axis("off")
axes[1].imshow(heatmap, cmap="jet"); axes[1].set_title("Grad-CAM"); axes[1].axis("off")
axes[2].imshow(overlay); axes[2].set_title("Overlay"); axes[2].axis("off")
plt.suptitle(f"Grad-CAM for finding: {taxonomy[top_idx]} (indicates contribution, not proof of pathology)")
plt.tight_layout(); plt.show()


## 15 — XAI: Integrated Gradients


In [ ]:
from xai import integrated_gradients as ig_mod

ig_map = ig_mod.compute_integrated_gradients(combo, input_tensor, target_class_idx=top_idx, n_steps=32)
ig_overlay = ig_mod.visualize_attributions(ig_map, demo_slice.astype(np.float32))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(ig_map, cmap="jet"); axes[0].set_title("Integrated Gradients"); axes[0].axis("off")
axes[1].imshow(ig_overlay); axes[1].set_title("Overlay"); axes[1].axis("off")
plt.tight_layout(); plt.show()


## 16 — XAI: Attention Visualization


In [ ]:
from xai import attention as attn_mod

attentions = attn_mod.extract_attention_maps(vision_encoder, input_tensor)
if attentions is None:
    print("This backbone does not expose attention weights (e.g. CNN fallback) -- skipping attention rollout.")
else:
    rollout = attn_mod.attention_rollout(attentions)
    attn_overlay = attn_mod.visualize_attention(rollout, demo_slice.astype(np.float32))
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(rollout, cmap="jet"); axes[0].set_title("Attention Rollout"); axes[0].axis("off")
    axes[1].imshow(attn_overlay); axes[1].set_title("Overlay"); axes[1].axis("off")
    plt.tight_layout(); plt.show()


### Occlusion Sensitivity


In [ ]:
from xai import occlusion as occ_mod

occ_map = occ_mod.occlusion_sensitivity(combo, input_tensor, target_class_idx=top_idx, patch_size=16, stride=8)
plt.figure(figsize=(4, 4))
plt.imshow(occ_map, cmap="jet")
plt.title("Occlusion Sensitivity")
plt.axis("off")
plt.show()


## 17 — Report Generation


In [ ]:
from reporting import schemas, findings as findings_mod, report_generator

raw_findings = [
    {"finding": name, "location": cfg.ANATOMICAL_REGIONS[0], "severity": "unknown",
     "model_score": float(score), "evidence": "Vision-model attribution score (research-demo, untrained head)."}
    for name, score in ranked
]
filtered = findings_mod.apply_confidence_filtering(raw_findings, min_confidence=cfg.MIN_CONFIDENCE)

xai_explanations = [
    schemas.XAIExplanation(
        finding=taxonomy[top_idx], method="Grad-CAM",
        important_regions=[cfg.ANATOMICAL_REGIONS[0]],
        explanation="Grad-CAM indicates the image region that contributed most strongly to the model's prediction.",
    )
]

study = schemas.StudyInfo(modality="MRI", body_region="brain", sequences=["Unknown"])
report = report_generator.generate_report_deterministic(filtered, xai_explanations, study, quality)
print(report_generator.render_text_report(report))

## 18 — Evaluation


In [ ]:
from evaluation import classification as eval_cls, segmentation as eval_seg, report_metrics as eval_txt

y_true = np.array([1, 0, 1, 1, 0])
y_pred = np.array([1, 0, 0, 1, 0])
y_score = np.array([0.9, 0.2, 0.4, 0.8, 0.3])
print("Classification metrics:", eval_cls.compute_classification_metrics(y_true, y_pred, y_score))

pred_mask = mask
gt_mask = (demo_slice > np.percentile(demo_slice, 90)).astype(np.uint8)
print("Dice:", eval_seg.dice_coefficient(pred_mask, gt_mask))
print("IoU:", eval_seg.iou_score(pred_mask, gt_mask))

print(eval_txt.TEXT_METRIC_CAVEAT)
sample_ref = "Mild left periventricular white matter signal abnormality."
sample_hyp = report.impression[0] if report.impression else ""
print("Text metrics (reference vs generated impression):", eval_txt.compute_text_metrics(sample_ref, sample_hyp))


## 19 — Gradio Application


In [ ]:
from ui.gradio_app import build_interface

demo = build_interface()
print("Gradio interface built. Launch in the final cell with demo.launch(share=True).")


## 20 — Model Export / Checkpoints


In [ ]:
checkpoint_path = cfg.CHECKPOINTS_DIR / "multimodal_projector.pt"
try:
    from models.multimodal_model import MultimodalProjector
    projector = MultimodalProjector(vision_encoder.hidden_dim, hidden_dim=cfg.MODEL_CONFIG["projection_hidden_dim"]).to(model_manager.device)
    torch.save(projector.state_dict(), checkpoint_path)
    print("Saved projector checkpoint to", checkpoint_path)
except Exception as exc:
    print("Checkpoint export skipped:", exc)

model_manager.unload_model(vision_encoder)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory cache cleared.")


## 21 — Final Demonstration

Launches the full Gradio interface: upload MRI -> select sequence/body region -> Analyze ->
view original + Grad-CAM heatmap + segmentation -> review findings with confidence -> read the
generated structured report -> download it.


In [ ]:
demo.launch(share=False, debug=False)
